In [ ]:
# =========================================================
# 2-Stage Demo: Deblur (PyTorch) + Colorize (Keras) in Gradio
# End-to-End tab: Stage1 -> Stage2 with smart grayscale check
# =========================================================
# pip install gradio==4.* pillow scikit-image opencv-python torch torchvision tensorflow

import os, numpy as np, gradio as gr
from PIL import Image

# ---------------- USER PATHS (edit as needed) ----------------
DEBLUR_WEIGHTS = "/kaggle/working/deblurganv2_brightness_best.pth"  # stage-1 .pth
DEBLUR_FPN_CH  = 128
DEBLUR_DEC_CH  = 128

COLOR_MODEL_PATH = "colorization_model.h5"  # stage-2 .h5/.keras
COLOR_IMG_SIZE   = 128                      # match your training size

# ---------------- Common helpers ----------------
def pil_to_uint8_rgb(img: Image.Image) -> np.ndarray:
    return np.array(img.convert("RGB"))

def uint8_to_pil(arr: np.ndarray) -> Image.Image:
    return Image.fromarray(arr.astype(np.uint8))

def is_grayscale(rgb_u8: np.ndarray, eps: float = 2.0) -> bool:
    """
    Quick grayscale detector: if channel differences are tiny on average.
    eps≈2 (on 0..255) is strict; raise slightly if needed.
    """
    r, g, b = rgb_u8[...,0].astype(np.float32), rgb_u8[...,1].astype(np.float32), rgb_u8[...,2].astype(np.float32)
    diff = np.maximum.reduce([np.abs(r-g), np.abs(r-b), np.abs(g-b)])
    return float(diff.mean()) <= eps

# =========================================================
#                   STAGE 1: DEBLUR (PyTorch)
# =========================================================
import torch, torch.nn as nn
import torch.nn.functional as F
from skimage import color

device_torch = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ResidualBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.c1 = nn.Conv2d(ch, ch, 3, padding=1)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(ch)
        self.bn2 = nn.BatchNorm2d(ch)
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        y = self.act(self.bn1(self.c1(x)))
        y = self.bn2(self.c2(y))
        return self.act(x + y)

class SimpleFPN(nn.Module):
    def __init__(self, in_ch=1, fpn_ch=128, dec_ch=128):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(in_ch, fpn_ch, 3, padding=1), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv2d(fpn_ch, fpn_ch, 3, stride=2, padding=1), nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv2d(fpn_ch, fpn_ch, 3, stride=2, padding=1), nn.ReLU(True))
        self.res  = nn.Sequential(ResidualBlock(fpn_ch), ResidualBlock(fpn_ch))
        self.up2  = nn.ConvTranspose2d(fpn_ch, dec_ch, 2, stride=2)
        self.up1  = nn.ConvTranspose2d(dec_ch, dec_ch, 2, stride=2)
        self.out  = nn.Conv2d(dec_ch, 1, 3, padding=1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        b  = self.res(e3)
        d2 = self.up2(b)
        d1 = self.up1(d2)
        y  = self.out(d1)
        return torch.sigmoid(y)

class DeblurGANv2Gen(nn.Module):
    def __init__(self, fpn_ch=128, dec_ch=128):
        super().__init__()
        self.net = SimpleFPN(in_ch=1, fpn_ch=fpn_ch, dec_ch=dec_ch)
    def forward(self, x):
        return self.net(x)

# Load generator
G = DeblurGANv2Gen(fpn_ch=DEBLUR_FPN_CH, dec_ch=DEBLUR_DEC_CH).to(device_torch)
if not os.path.exists(DEBLUR_WEIGHTS):
    raise FileNotFoundError(f"Deblur weights not found: {DEBLUR_WEIGHTS}")
G.load_state_dict(torch.load(DEBLUR_WEIGHTS, map_location=device_torch))
G.eval()

def rgb_to_L01(rgb_uint8: np.ndarray) -> torch.Tensor:
    arr = rgb_uint8.astype(np.float32) / 255.0
    Lab = color.rgb2lab(arr).astype(np.float32)
    L01 = np.clip(Lab[...,0] / 100.0, 0, 1)
    return torch.from_numpy(L01).unsqueeze(0).unsqueeze(0).float()

def compose_L_with_original_ab(L_restored01: torch.Tensor, rgb_uint8: np.ndarray) -> np.ndarray:
    L = (L_restored01.squeeze().cpu().numpy() * 100.0).clip(0,100).astype(np.float32)
    arr = rgb_uint8.astype(np.float32) / 255.0
    Lab_in = color.rgb2lab(arr).astype(np.float32)
    Lab_out = np.dstack([L, Lab_in[...,1], Lab_in[...,2]])
    rgb = (color.lab2rgb(Lab_out) * 255.0).clip(0,255).astype(np.uint8)
    return rgb

def run_deblur_pipeline(input_img: Image.Image) -> Image.Image:
    rgb_u8 = pil_to_uint8_rgb(input_img)
    H, W = rgb_u8.shape[:2]
    MAX_SIDE = 1024
    if max(H, W) > MAX_SIDE:
        scale = MAX_SIDE / max(H, W)
        rgb_u8 = np.array(Image.fromarray(rgb_u8).resize((int(W*scale), int(H*scale)), Image.BICUBIC))
    L01 = rgb_to_L01(rgb_u8).to(device_torch)
    with torch.no_grad(), torch.amp.autocast(device_type="cuda", enabled=(device_torch.type=="cuda")):
        L_restored = G(L01)
    rgb_restored = compose_L_with_original_ab(L_restored, rgb_u8)
    return uint8_to_pil(rgb_restored)

# =========================================================
#                STAGE 2: COLORIZE (Keras)
# =========================================================
import tensorflow as tf
from tensorflow.keras.models import load_model

if not os.path.exists(COLOR_MODEL_PATH):
    raise FileNotFoundError(f"Colorization model not found: {COLOR_MODEL_PATH}")
color_net = load_model(COLOR_MODEL_PATH, compile=False)

def preprocess_gray_for_color(rgb_uint8: np.ndarray, size=COLOR_IMG_SIZE):
    im = Image.fromarray(rgb_uint8).resize((size, size), Image.BICUBIC)
    rgb = np.asarray(im).astype(np.float32) / 255.0
    gray = np.dot(rgb[..., :3], [0.2989, 0.5870, 0.1140])[..., None]
    return gray[None, ...]  # [1,H,W,1]

def run_colorize_pipeline(input_img: Image.Image):
    rgb_u8 = pil_to_uint8_rgb(input_img)
    gray_b = preprocess_gray_for_color(rgb_u8, COLOR_IMG_SIZE)   # [1,H,W,1]
    pred = color_net.predict(gray_b, verbose=0)[0]               # [H,W,3] in [0,1]
    pred_u8 = (pred * 255.0).clip(0,255).astype(np.uint8)
    return uint8_to_pil(pred_u8)

# =========================================================
#                END-TO-END: Stage1 → Stage2
# =========================================================
def run_end_to_end(input_img: Image.Image):
    """
    1) Deblur (brightness) → RGB deblurred.
    2) If deblurred is grayscale → colorize. Else, return deblurred as-is.
    """
    if input_img is None:
        return None, "Please upload an image."
    # Stage 1
    deblurred = run_deblur_pipeline(input_img)  # PIL
    deb_u8 = pil_to_uint8_rgb(deblurred)

    # Grayscale check on deblurred output
    if is_grayscale(deb_u8, eps=2.0):
        colored = run_colorize_pipeline(deblurred)
        return colored, "Detected grayscale → colorized after deblurring."
    else:
        return deblurred, "Detected color → returned deblurred image."

# =========================================================
#                      GRADIO UI
# =========================================================
with gr.Blocks(title="Two-Stage: Deblur + Colorize") as demo:
    gr.Markdown("## Two-Stage Image Pipeline")
    gr.Markdown(
        f"- **Stage 1 (Restoration)** — PyTorch deblurring on brightness channel (weights: `{os.path.basename(DEBLUR_WEIGHTS)}`).\n"
        f"- **Stage 2 (Colorization)** — Keras U-Net colorizer (model: `{os.path.basename(COLOR_MODEL_PATH)}`, size: {COLOR_IMG_SIZE}).\n"
        f"- **End-to-End** — Runs Stage 1 → Stage 2 automatically; only colorizes if result is grayscale."
    )

    with gr.Tab("Stage 1 — Deblur"):
        with gr.Row():
            s1_in  = gr.Image(type="pil", label="Input image", height=300)
            s1_out = gr.Image(type="pil", label="Deblurred (RGB)", height=300)
        gr.Button("Run Deblur").click(fn=run_deblur_pipeline, inputs=s1_in, outputs=s1_out)

    with gr.Tab("Stage 2 — Colorize"):
        with gr.Row():
            s2_in  = gr.Image(type="pil", label="Input (any image)", height=300)
            s2_out = gr.Image(type="pil", label="Colorized output", height=300)
        gr.Button("Run Colorization").click(fn=run_colorize_pipeline, inputs=s2_in, outputs=s2_out)

    with gr.Tab("End-to-End — Deblur → Colorize if Gray"):
        with gr.Row():
            e2e_in   = gr.Image(type="pil", label="Input image", height=300)
            e2e_out  = gr.Image(type="pil", label="Final Output", height=300)
        e2e_msg = gr.Markdown("")
        gr.Button("Run Full Pipeline").click(fn=run_end_to_end, inputs=e2e_in, outputs=[e2e_out, e2e_msg])

demo.launch()
